In [8]:
# ==========================================================
# Cell 1 - Privacy Classifier Configuration
# ==========================================================

import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader


# ----------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# ----------------------------------------------------------
# Device
# ----------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ----------------------------------------------------------
# Run mode
# ----------------------------------------------------------
# small_cpu:
#     Uses a small balanced subset for quick local testing.
#
# full_gpu:
#     Uses the complete multi-subject dataset on Narval.
# ----------------------------------------------------------

RUN_MODE = os.environ.get(
    "RUN_MODE",
    "small_cpu"
)

if RUN_MODE not in {
    "small_cpu",
    "full_gpu"
}:
    raise ValueError(
        "RUN_MODE must be either "
        "'small_cpu' or 'full_gpu'."
    )


# ----------------------------------------------------------
# Processed dataset directory
# ----------------------------------------------------------

PROCESSED_DIR = Path(
    os.environ.get(
        "IMU_PROCESSED_PATH",
        "../datasets/"
        "DatasetIMUandBIOMARKERS/"
        "processed"
    )
).expanduser().resolve()

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        "Processed dataset directory was not found:\n"
        f"{PROCESSED_DIR}"
    )


# ----------------------------------------------------------
# Select the private attribute
# ----------------------------------------------------------
# Available choices:
#     "gender" -> binary classification, 2 classes
#     "weight" -> three-class classification
# ----------------------------------------------------------

PRIVATE_ATTRIBUTE = os.environ.get(
    "PRIVATE_ATTRIBUTE",
    "gender"
).lower()

if PRIVATE_ATTRIBUTE == "gender":
    NUM_PRIVATE_CLASSES = 2

elif PRIVATE_ATTRIBUTE == "weight":
    NUM_PRIVATE_CLASSES = 3

else:
    raise ValueError(
        "PRIVATE_ATTRIBUTE must be either "
        "'gender' or 'weight'."
    )


# ----------------------------------------------------------
# Model and training configuration
# ----------------------------------------------------------

Z_DIM = 60

if RUN_MODE == "small_cpu":

    BATCH_SIZE = 8
    EPOCHS = 5
    LEARNING_RATE = 0.001

else:

    BATCH_SIZE = 64
    EPOCHS = 30
    LEARNING_RATE = 0.001


# ----------------------------------------------------------
# Configuration summary
# ----------------------------------------------------------

print("=" * 70)
print("PRIVACY CLASSIFIER CONFIGURATION")
print("=" * 70)

print("Run mode              :", RUN_MODE)
print("Device                :", device)
print("Private attribute     :", PRIVATE_ATTRIBUTE)
print("Private classes       :", NUM_PRIVATE_CLASSES)
print("Embedding dimension   :", Z_DIM)
print("Batch size            :", BATCH_SIZE)
print("Epochs                :", EPOCHS)
print("Learning rate         :", LEARNING_RATE)
print("Processed directory   :", PROCESSED_DIR)

PRIVACY CLASSIFIER CONFIGURATION
Run mode              : small_cpu
Device                : cpu
Private attribute     : gender
Private classes       : 2
Embedding dimension   : 60
Batch size            : 8
Epochs                : 5
Learning rate         : 0.001
Processed directory   : C:\PrivDiffuser_Narval\datasets\DatasetIMUandBIOMARKERS\processed


In [10]:
# ==========================================================
# Cell 2 - Balanced Dataset for Gender Privacy Classifier
# ==========================================================

import gc
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader


PROCESSED_DIR = Path(PROCESSED_DIR)

NUM_ACTIVITIES = 6
NUM_GENDERS = 2
NUM_WEIGHT_CLASSES = 3


def find_subjects_by_gender(processed_directory, split):
    """
    Find available subject files and group them according
    to their gender label.
    """

    files_by_gender = {
        0: [],
        1: []
    }

    subject_files = sorted(
        processed_directory.glob(f"*_{split}.npz")
    )

    for file_path in subject_files:

        with np.load(file_path, allow_pickle=False) as data:
            gender_value = int(
                np.asarray(data["gender"]).reshape(-1)[0]
            )

        if gender_value in files_by_gender:
            files_by_gender[gender_value].append(file_path)

    return files_by_gender


train_files_by_gender = find_subjects_by_gender(
    PROCESSED_DIR,
    split="train"
)

test_files_by_gender = find_subjects_by_gender(
    PROCESSED_DIR,
    split="test"
)


print("Available training subjects:")
print("Gender 0:", len(train_files_by_gender[0]))
print("Gender 1:", len(train_files_by_gender[1]))

print("\nAvailable testing subjects:")
print("Gender 0:", len(test_files_by_gender[0]))
print("Gender 1:", len(test_files_by_gender[1]))


# Use two training subjects from each gender
selected_train_files = (
    train_files_by_gender[0][:2]
    + train_files_by_gender[1][:2]
)

# Use one held-out testing subject from each gender
selected_test_files = (
    test_files_by_gender[0][:1]
    + test_files_by_gender[1][:1]
)


print("\nSelected training subjects:")
for file_path in selected_train_files:
    print(file_path.name)

print("\nSelected testing subjects:")
for file_path in selected_test_files:
    print(file_path.name)


def create_privacy_dataset(
    subject_files,
    samples_per_activity=20,
    seed=42
):
    """
    Create a small multi-subject dataset for gender prediction.

    An equal number of windows is selected from each activity
    for every chosen participant.
    """

    rng = np.random.default_rng(seed)

    all_windows = []
    all_activities = []
    all_genders = []
    all_weights = []

    for file_path in subject_files:

        with np.load(file_path, allow_pickle=False) as data:

            activity_labels = np.asarray(
                data["activity"],
                dtype=np.int64
            )

            gender_value = int(
                np.asarray(data["gender"]).reshape(-1)[0]
            )

            weight_value = int(
                np.asarray(data["weight"]).reshape(-1)[0]
            )

            selected_indices = []

            for activity_id in range(NUM_ACTIVITIES):

                activity_indices = np.flatnonzero(
                    activity_labels == activity_id
                )

                if len(activity_indices) == 0:
                    continue

                number_to_select = min(
                    samples_per_activity,
                    len(activity_indices)
                )

                chosen_indices = rng.choice(
                    activity_indices,
                    size=number_to_select,
                    replace=False
                )

                selected_indices.extend(
                    chosen_indices.tolist()
                )

            selected_indices = np.asarray(
                selected_indices,
                dtype=np.int64
            )

            rng.shuffle(selected_indices)

            # Decompress the large windows array only once
            subject_windows = data["windows"]

            selected_windows = np.asarray(
                subject_windows[selected_indices],
                dtype=np.float32
            ).copy()

            selected_activities = activity_labels[
                selected_indices
            ].copy()

        number_of_selected_windows = len(selected_indices)

        all_windows.append(selected_windows)
        all_activities.append(selected_activities)

        all_genders.append(
            np.full(
                number_of_selected_windows,
                gender_value,
                dtype=np.int64
            )
        )

        all_weights.append(
            np.full(
                number_of_selected_windows,
                weight_value,
                dtype=np.int64
            )
        )

        print(
            f"Loaded {file_path.name}: "
            f"{number_of_selected_windows} windows, "
            f"gender={gender_value}"
        )

        del subject_windows
        del selected_windows
        gc.collect()

    windows_array = np.concatenate(
        all_windows,
        axis=0
    )

    activity_array = np.concatenate(
        all_activities,
        axis=0
    )

    gender_array = np.concatenate(
        all_genders,
        axis=0
    )

    weight_array = np.concatenate(
        all_weights,
        axis=0
    )

    # Shuffle the complete multi-subject dataset
    shuffle_indices = rng.permutation(
        len(windows_array)
    )

    windows_array = windows_array[shuffle_indices]
    activity_array = activity_array[shuffle_indices]
    gender_array = gender_array[shuffle_indices]
    weight_array = weight_array[shuffle_indices]

    windows_tensor = torch.from_numpy(
        windows_array
    ).unsqueeze(1)

    activity_tensor = F.one_hot(
        torch.from_numpy(activity_array),
        num_classes=NUM_ACTIVITIES
    ).float()

    gender_tensor = F.one_hot(
        torch.from_numpy(gender_array),
        num_classes=NUM_GENDERS
    ).float()

    weight_tensor = F.one_hot(
        torch.from_numpy(weight_array),
        num_classes=NUM_WEIGHT_CLASSES
    ).float()

    print(
        "\nGender counts:",
        torch.bincount(
            torch.from_numpy(gender_array),
            minlength=NUM_GENDERS
        ).tolist()
    )

    return TensorDataset(
        windows_tensor,
        activity_tensor,
        gender_tensor,
        weight_tensor
    )


# Four training subjects:
# 4 × 6 activities × 20 windows = approximately 480 samples
privacy_train_dataset = create_privacy_dataset(
    selected_train_files,
    samples_per_activity=20,
    seed=42
)

# Two held-out testing subjects:
# 2 × 6 activities × 20 windows = approximately 240 samples
privacy_test_dataset = create_privacy_dataset(
    selected_test_files,
    samples_per_activity=20,
    seed=43
)


privacy_train_loader = DataLoader(
    privacy_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

privacy_test_loader = DataLoader(
    privacy_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


print("\nPrivacy dataset successfully created")
print("Privacy training samples:", len(privacy_train_dataset))
print("Privacy testing samples :", len(privacy_test_dataset))


privacy_batch = next(
    iter(privacy_train_loader)
)

privacy_windows, privacy_activity, privacy_gender, privacy_weight = (
    privacy_batch
)

print("\nOne privacy training batch:")
print("Windows shape :", tuple(privacy_windows.shape))
print("Activity shape:", tuple(privacy_activity.shape))
print("Gender shape  :", tuple(privacy_gender.shape))
print("Weight shape  :", tuple(privacy_weight.shape))

Available training subjects:
Gender 0: 28
Gender 1: 20

Available testing subjects:
Gender 0: 8
Gender 1: 4

Selected training subjects:
Subject02_train.npz
Subject07_train.npz
Subject03_train.npz
Subject04_train.npz

Selected testing subjects:
Subject01_test.npz
Subject54_test.npz
Loaded Subject02_train.npz: 120 windows, gender=0
Loaded Subject07_train.npz: 120 windows, gender=0
Loaded Subject03_train.npz: 120 windows, gender=1
Loaded Subject04_train.npz: 120 windows, gender=1

Gender counts: [240, 240]
Loaded Subject01_test.npz: 120 windows, gender=0
Loaded Subject54_test.npz: 120 windows, gender=1

Gender counts: [120, 120]

Privacy dataset successfully created
Privacy training samples: 480
Privacy testing samples : 240

One privacy training batch:
Windows shape : (8, 1, 128, 30)
Activity shape: (8, 6)
Gender shape  : (8, 2)
Weight shape  : (8, 3)


In [11]:
# ==========================================================
# Cell 3 - Privacy Classifier for CPU and GPU
# ==========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class PrivacyClassifier(nn.Module):
    """
    Lightweight CNN for private-attribute prediction.

    Input:
        IMU windows with shape:
        (batch_size, 1, 128, 30)

    Outputs:
        private_logits:
            Raw logits for gender or weight prediction.

        z_private:
            Private representation with dimension Z_DIM.
    """

    def __init__(
        self,
        num_private_classes,
        z_dim
    ):
        super().__init__()

        self.feature_extractor = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2
            ),

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(
                output_size=(1, 1)
            )
        )

        self.private_embedding = nn.Linear(
            in_features=32,
            out_features=z_dim
        )

        self.private_classifier = nn.Linear(
            in_features=z_dim,
            out_features=num_private_classes
        )

    def forward(self, x):

        features = self.feature_extractor(x)

        features = torch.flatten(
            features,
            start_dim=1
        )

        z_private = F.relu(
            self.private_embedding(features)
        )

        private_logits = self.private_classifier(
            z_private
        )

        return private_logits, z_private


# ==========================================================
# Create Privacy Model
# ==========================================================

privacy_model = PrivacyClassifier(
    num_private_classes=NUM_PRIVATE_CLASSES,
    z_dim=Z_DIM
).to(device)


print("=" * 70)
print("PRIVACY CLASSIFIER")
print("=" * 70)

print("Run mode          :", RUN_MODE)
print("Device            :", device)
print("Private attribute :", PRIVATE_ATTRIBUTE)
print("Private classes   :", NUM_PRIVATE_CLASSES)
print("Embedding size    :", Z_DIM)


# ==========================================================
# Select Verification Loader
# ==========================================================

if RUN_MODE == "small_cpu":

    verification_loader = privacy_train_loader

else:

    # For full GPU mode, this loader must be created from
    # the full multi-subject privacy dataset.
    verification_loader = full_privacy_train_loader


# ==========================================================
# Verify One Forward Pass
# ==========================================================

privacy_windows, privacy_activity, privacy_gender, privacy_weight = next(
    iter(verification_loader)
)

privacy_model.eval()

with torch.no_grad():

    privacy_windows_device = privacy_windows.to(
        device,
        dtype=torch.float32,
        non_blocking=True
    )

    test_private_logits, test_private_embedding = privacy_model(
        privacy_windows_device
    )


assert test_private_logits.shape == (
    privacy_windows.shape[0],
    NUM_PRIVATE_CLASSES
)

assert test_private_embedding.shape == (
    privacy_windows.shape[0],
    Z_DIM
)


print("\n" + "=" * 70)
print("PRIVACY MODEL OUTPUT VERIFICATION")
print("=" * 70)

print(
    "Input window shape      :",
    tuple(privacy_windows.shape)
)

print(
    "Private logits shape    :",
    tuple(test_private_logits.shape)
)

print(
    "Private embedding shape :",
    tuple(test_private_embedding.shape)
)

print(
    "\nPrivacy classifier verified successfully."
)

PRIVACY CLASSIFIER
Run mode          : small_cpu
Device            : cpu
Private attribute : gender
Private classes   : 2
Embedding size    : 60

PRIVACY MODEL OUTPUT VERIFICATION
Input window shape      : (8, 1, 128, 30)
Private logits shape    : (8, 2)
Private embedding shape : (8, 60)

Privacy classifier verified successfully.


In [12]:
# ==========================================================
# Cell 4 - Train Privacy Classifier
# CPU and GPU/Narval Compatible
# ==========================================================

import time
from pathlib import Path

import torch
import torch.nn as nn


# ----------------------------------------------------------
# Select training loader
# ----------------------------------------------------------

if RUN_MODE == "small_cpu":

    privacy_training_loader = privacy_train_loader

else:

    # This loader must be created from the full
    # multi-subject privacy dataset before full GPU training.
    privacy_training_loader = full_privacy_train_loader


# ----------------------------------------------------------
# Loss function and optimizer
# ----------------------------------------------------------

privacy_criterion = nn.CrossEntropyLoss()

privacy_optimizer = torch.optim.Adam(
    privacy_model.parameters(),
    lr=LEARNING_RATE
)

PRIVACY_EPOCHS = EPOCHS


# ----------------------------------------------------------
# Store training history
# ----------------------------------------------------------

privacy_training_losses = []
privacy_training_accuracies = []

start_time = time.time()


print("=" * 75)
print("PRIVACY CLASSIFIER TRAINING")
print("=" * 75)

print("Run mode          :", RUN_MODE)
print("Device            :", device)
print("Private attribute :", PRIVATE_ATTRIBUTE)
print("Epochs            :", PRIVACY_EPOCHS)
print("Learning rate     :", LEARNING_RATE)
print("Batch size        :", BATCH_SIZE)


# ----------------------------------------------------------
# Training loop
# ----------------------------------------------------------

for epoch in range(PRIVACY_EPOCHS):

    privacy_model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for windows, activity, gender, weight in privacy_training_loader:

        windows = windows.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # --------------------------------------------------
        # Select the private target
        # --------------------------------------------------

        if PRIVATE_ATTRIBUTE == "gender":
            private_labels = gender

        else:
            private_labels = weight


        # Small CPU datasets currently return one-hot labels.
        # Full GPU datasets may return integer class labels.
        if private_labels.ndim > 1:

            private_targets = torch.argmax(
                private_labels,
                dim=1
            )

        else:

            private_targets = private_labels


        private_targets = private_targets.to(
            device,
            dtype=torch.long,
            non_blocking=True
        )


        # --------------------------------------------------
        # Forward and backward pass
        # --------------------------------------------------

        privacy_optimizer.zero_grad(
            set_to_none=True
        )

        private_logits, z_private = privacy_model(
            windows
        )

        loss = privacy_criterion(
            private_logits,
            private_targets
        )

        loss.backward()
        privacy_optimizer.step()


        # --------------------------------------------------
        # Training statistics
        # --------------------------------------------------

        batch_size = private_targets.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        predicted_private_classes = torch.argmax(
            private_logits,
            dim=1
        )

        correct_predictions += (
            predicted_private_classes
            == private_targets
        ).sum().item()

        total_samples += batch_size


    epoch_loss = (
        running_loss / total_samples
    )

    epoch_accuracy = (
        correct_predictions / total_samples
    )

    privacy_training_losses.append(
        epoch_loss
    )

    privacy_training_accuracies.append(
        epoch_accuracy
    )

    print(
        f"Epoch {epoch + 1:02d}/{PRIVACY_EPOCHS:02d} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Training Accuracy: {epoch_accuracy:.4f}"
    )


# ----------------------------------------------------------
# Training summary
# ----------------------------------------------------------

training_time = (
    time.time() - start_time
)

print("\n" + "=" * 75)
print("PRIVACY CLASSIFIER TRAINING COMPLETED")
print("=" * 75)

print(
    f"Final training loss     : "
    f"{privacy_training_losses[-1]:.4f}"
)

print(
    f"Final training accuracy : "
    f"{privacy_training_accuracies[-1]:.4f}"
)

print(
    f"Total training time     : "
    f"{training_time:.2f} seconds"
)


# ----------------------------------------------------------
# Save model checkpoint
# ----------------------------------------------------------

model_filename = (
    f"{RUN_MODE}_privacy_"
    f"{PRIVATE_ATTRIBUTE}_model.pt"
)

MODEL_SAVE_PATH = Path(
    model_filename
)

checkpoint = {
    "model_state_dict":
        privacy_model.state_dict(),

    "optimizer_state_dict":
        privacy_optimizer.state_dict(),

    "training_losses":
        privacy_training_losses,

    "training_accuracies":
        privacy_training_accuracies,

    "epochs":
        PRIVACY_EPOCHS,

    "learning_rate":
        LEARNING_RATE,

    "private_attribute":
        PRIVATE_ATTRIBUTE,

    "num_private_classes":
        NUM_PRIVATE_CLASSES,

    "z_dim":
        Z_DIM,

    "run_mode":
        RUN_MODE
}

torch.save(
    checkpoint,
    MODEL_SAVE_PATH
)

print(
    "\nPrivacy model checkpoint saved as:",
    MODEL_SAVE_PATH
)

PRIVACY CLASSIFIER TRAINING
Run mode          : small_cpu
Device            : cpu
Private attribute : gender
Epochs            : 5
Learning rate     : 0.001
Batch size        : 8
Epoch 01/05 | Loss: 0.6974 | Training Accuracy: 0.5000
Epoch 02/05 | Loss: 0.6926 | Training Accuracy: 0.5083
Epoch 03/05 | Loss: 0.6920 | Training Accuracy: 0.5146
Epoch 04/05 | Loss: 0.6910 | Training Accuracy: 0.5604
Epoch 05/05 | Loss: 0.7001 | Training Accuracy: 0.4917

PRIVACY CLASSIFIER TRAINING COMPLETED
Final training loss     : 0.7001
Final training accuracy : 0.4917
Total training time     : 1.58 seconds

Privacy model checkpoint saved as: small_cpu_privacy_gender_model.pt


In [13]:
# ==========================================================
# Cell 5 - Evaluate Privacy Classifier
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ----------------------------------------------------------
# Select evaluation loader and dataset sizes
# ----------------------------------------------------------

if RUN_MODE == "small_cpu":

    privacy_evaluation_loader = privacy_test_loader

    training_sample_count = len(
        privacy_train_dataset
    )

    testing_sample_count = len(
        privacy_test_dataset
    )

else:

    privacy_evaluation_loader = (
        full_privacy_test_loader
    )

    training_sample_count = len(
        full_privacy_train_loader.dataset
    )

    testing_sample_count = len(
        full_privacy_test_loader.dataset
    )


# ----------------------------------------------------------
# Select class names
# ----------------------------------------------------------

if PRIVATE_ATTRIBUTE == "gender":

    private_class_names = [
        "Gender 0",
        "Gender 1"
    ]

    private_class_ids = [
        0,
        1
    ]

else:

    private_class_names = [
        "Weight Class 0",
        "Weight Class 1",
        "Weight Class 2"
    ]

    private_class_ids = [
        0,
        1,
        2
    ]


# ----------------------------------------------------------
# Run evaluation
# ----------------------------------------------------------

privacy_model.eval()

true_private_labels = []
predicted_private_labels = []

with torch.no_grad():

    for windows, activity, gender, weight in (
        privacy_evaluation_loader
    ):

        windows = windows.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # Select the sensitive target
        if PRIVATE_ATTRIBUTE == "gender":
            private_labels = gender
        else:
            private_labels = weight

        # Small CPU datasets use one-hot labels.
        # Full GPU datasets may use integer labels.
        if private_labels.ndim > 1:

            private_targets = torch.argmax(
                private_labels,
                dim=1
            )

        else:

            private_targets = private_labels

        private_targets = private_targets.to(
            device,
            dtype=torch.long,
            non_blocking=True
        )

        private_logits, z_private = privacy_model(
            windows
        )

        predicted_classes = torch.argmax(
            private_logits,
            dim=1
        )

        true_private_labels.extend(
            private_targets.cpu().numpy()
        )

        predicted_private_labels.extend(
            predicted_classes.cpu().numpy()
        )


true_private_labels = np.asarray(
    true_private_labels
)

predicted_private_labels = np.asarray(
    predicted_private_labels
)


# ----------------------------------------------------------
# Calculate evaluation metrics
# ----------------------------------------------------------

privacy_accuracy = accuracy_score(
    true_private_labels,
    predicted_private_labels
)

privacy_precision = precision_score(
    true_private_labels,
    predicted_private_labels,
    average="macro",
    zero_division=0
)

privacy_recall = recall_score(
    true_private_labels,
    predicted_private_labels,
    average="macro",
    zero_division=0
)

privacy_macro_f1 = f1_score(
    true_private_labels,
    predicted_private_labels,
    average="macro",
    zero_division=0
)

privacy_weighted_f1 = f1_score(
    true_private_labels,
    predicted_private_labels,
    average="weighted",
    zero_division=0
)


# ----------------------------------------------------------
# Print overall results
# ----------------------------------------------------------

print("=" * 75)
print("PRIVACY CLASSIFICATION RESULTS")
print("=" * 75)

print("Run mode            :", RUN_MODE)
print("Device              :", device)
print("Private attribute   :", PRIVATE_ATTRIBUTE)
print("Training samples    :", training_sample_count)
print("Testing samples     :", testing_sample_count)
print(f"Test Accuracy       : {privacy_accuracy:.4f}")
print(f"Macro Precision     : {privacy_precision:.4f}")
print(f"Macro Recall        : {privacy_recall:.4f}")
print(f"Macro F1-score      : {privacy_macro_f1:.4f}")
print(f"Weighted F1-score   : {privacy_weighted_f1:.4f}")


# ----------------------------------------------------------
# Classification report
# ----------------------------------------------------------

print("\n" + "=" * 75)
print("PER-CLASS PRIVACY CLASSIFICATION REPORT")
print("=" * 75)

print(
    classification_report(
        true_private_labels,
        predicted_private_labels,
        labels=private_class_ids,
        target_names=private_class_names,
        digits=4,
        zero_division=0
    )
)


# ----------------------------------------------------------
# Plain confusion matrix
# ----------------------------------------------------------

privacy_cm = confusion_matrix(
    true_private_labels,
    predicted_private_labels,
    labels=private_class_ids
)

confusion_matrix_table = pd.DataFrame(
    privacy_cm,
    index=[
        f"True {name}"
        for name in private_class_names
    ],
    columns=[
        f"Predicted {name}"
        for name in private_class_names
    ]
)

print("=" * 85)
print("PRIVACY CONFUSION MATRIX")
print("=" * 85)

print(
    confusion_matrix_table.to_string()
)


# ----------------------------------------------------------
# Chance-level reference
# ----------------------------------------------------------

chance_accuracy = (
    1.0 / NUM_PRIVATE_CLASSES
)

print(
    f"\nChance-level accuracy : "
    f"{chance_accuracy:.4f}"
)

PRIVACY CLASSIFICATION RESULTS
Run mode            : small_cpu
Device              : cpu
Private attribute   : gender
Training samples    : 480
Testing samples     : 240
Test Accuracy       : 0.4958
Macro Precision     : 0.2490
Macro Recall        : 0.4958
Macro F1-score      : 0.3315
Weighted F1-score   : 0.3315

PER-CLASS PRIVACY CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Gender 0     0.0000    0.0000    0.0000       120
    Gender 1     0.4979    0.9917    0.6630       120

    accuracy                         0.4958       240
   macro avg     0.2490    0.4958    0.3315       240
weighted avg     0.2490    0.4958    0.3315       240

PRIVACY CONFUSION MATRIX
               Predicted Gender 0  Predicted Gender 1
True Gender 0                   0                 120
True Gender 1                   1                 119

Chance-level accuracy : 0.5000


In [16]:
# ==========================================================
# Full Narval Privacy Dataset
# Balanced, Memory-Efficient, Multi-Subject Loader
# ==========================================================

from pathlib import Path
import os

import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader


class BalancedPrivacyIMUDataset(Dataset):
    """
    Memory-efficient, subject-wise balanced dataset for
    private-attribute prediction.

    Balancing is performed at three levels:

    1. Equal number of subjects from each private class.
    2. Equal number of windows from each activity.
    3. Equal contribution from every selected subject.

    Each item returns:
        window:
            float32 tensor with shape (1, 128, 30)

        activity:
            integer label from 0 to 5

        gender:
            integer label from 0 to 1

        weight:
            integer label from 0 to 2
    """

    def __init__(
        self,
        processed_dir,
        split="train",
        private_attribute="gender",
        samples_per_activity_per_subject=500,
        seed=42
    ):
        super().__init__()

        self.processed_dir = Path(
            processed_dir
        ).expanduser().resolve()

        self.split = split

        self.private_attribute = (
            private_attribute.lower()
        )

        self.requested_samples = int(
            samples_per_activity_per_subject
        )

        self.seed = seed

        self.rng = np.random.default_rng(
            seed
        )

        # --------------------------------------------------
        # Validate configuration
        # --------------------------------------------------

        if self.split not in {
            "train",
            "test"
        }:
            raise ValueError(
                "split must be either "
                "'train' or 'test'."
            )

        if self.private_attribute not in {
            "gender",
            "weight"
        }:
            raise ValueError(
                "private_attribute must be either "
                "'gender' or 'weight'."
            )

        if not self.processed_dir.exists():
            raise FileNotFoundError(
                "Processed directory was not found:\n"
                f"{self.processed_dir}"
            )

        if self.requested_samples <= 0:
            raise ValueError(
                "samples_per_activity_per_subject "
                "must be greater than zero."
            )

        self.num_private_classes = (
            2
            if self.private_attribute == "gender"
            else 3
        )

        self.num_activities = 6

        # --------------------------------------------------
        # Find processed subject files
        # --------------------------------------------------

        all_files = sorted(
            self.processed_dir.glob(
                f"*_{self.split}.npz"
            )
        )

        if not all_files:
            raise FileNotFoundError(
                f"No '*_{self.split}.npz' files "
                f"were found in:\n"
                f"{self.processed_dir}"
            )

        # --------------------------------------------------
        # Group subject files by private class
        # --------------------------------------------------

        files_by_private_class = {
            class_id: []
            for class_id in range(
                self.num_private_classes
            )
        }

        file_information = {}

        for file_path in all_files:

            with np.load(
                file_path,
                allow_pickle=False
            ) as data:

                required_keys = {
                    "windows",
                    "activity",
                    "gender",
                    "weight"
                }

                missing_keys = (
                    required_keys
                    - set(data.files)
                )

                if missing_keys:
                    raise KeyError(
                        f"{file_path.name} is missing "
                        f"the following arrays: "
                        f"{sorted(missing_keys)}"
                    )

                if data["windows"].ndim != 3:
                    raise ValueError(
                        f"Expected a three-dimensional "
                        f"window array in "
                        f"{file_path.name}, but found "
                        f"{data['windows'].shape}."
                    )

                if data["windows"].shape[1:] != (
                    128,
                    30
                ):
                    raise ValueError(
                        f"Unexpected window shape in "
                        f"{file_path.name}: "
                        f"{data['windows'].shape}"
                    )

                if (
                    len(data["windows"])
                    != len(data["activity"])
                ):
                    raise ValueError(
                        f"Window-label count mismatch in "
                        f"{file_path.name}."
                    )

                activity_labels = np.asarray(
                    data["activity"],
                    dtype=np.int64
                )

                gender_value = int(
                    np.asarray(
                        data["gender"]
                    ).reshape(-1)[0]
                )

                weight_value = int(
                    np.asarray(
                        data["weight"]
                    ).reshape(-1)[0]
                )

                private_value = (
                    gender_value
                    if self.private_attribute == "gender"
                    else weight_value
                )

                if private_value not in (
                    files_by_private_class
                ):
                    raise ValueError(
                        f"Invalid "
                        f"{self.private_attribute} label "
                        f"{private_value} in "
                        f"{file_path.name}."
                    )

                activity_counts = {
                    activity_id: int(
                        np.sum(
                            activity_labels
                            == activity_id
                        )
                    )
                    for activity_id
                    in range(
                        self.num_activities
                    )
                }

                files_by_private_class[
                    private_value
                ].append(
                    file_path
                )

                file_information[
                    file_path
                ] = {
                    "private_value":
                        private_value,

                    "gender":
                        gender_value,

                    "weight":
                        weight_value,

                    "activity_counts":
                        activity_counts
                }

        # --------------------------------------------------
        # Balance subject counts across private classes
        # --------------------------------------------------

        available_subject_counts = {
            class_id: len(class_files)
            for class_id, class_files
            in files_by_private_class.items()
        }

        subjects_per_class = min(
            available_subject_counts.values()
        )

        if subjects_per_class == 0:
            raise ValueError(
                "At least one private class contains "
                "no subject files."
            )

        selected_files = []

        for class_id in range(
            self.num_private_classes
        ):

            class_files = (
                files_by_private_class[
                    class_id
                ]
            )

            selected_positions = (
                self.rng.choice(
                    len(class_files),
                    size=subjects_per_class,
                    replace=False
                )
            )

            selected_class_files = [
                class_files[
                    int(position)
                ]
                for position
                in selected_positions
            ]

            selected_files.extend(
                selected_class_files
            )

        self.files = sorted(
            selected_files
        )

        # --------------------------------------------------
        # Determine common sample count
        # --------------------------------------------------
        # A common number is used across every selected
        # subject and activity. This produces strict balance.
        # --------------------------------------------------

        available_activity_counts = []

        for file_path in self.files:

            for activity_id in range(
                self.num_activities
            ):

                count = file_information[
                    file_path
                ]["activity_counts"][
                    activity_id
                ]

                if count == 0:
                    raise ValueError(
                        f"{file_path.name} contains no "
                        f"windows for Activity "
                        f"{activity_id}."
                    )

                available_activity_counts.append(
                    count
                )

        minimum_available_count = min(
            available_activity_counts
        )

        self.samples_per_activity_per_subject = min(
            self.requested_samples,
            minimum_available_count
        )

        if (
            self.samples_per_activity_per_subject
            < self.requested_samples
        ):
            print(
                "\nWarning: The requested number of "
                "windows could not be used for every "
                "subject and activity."
            )

            print(
                "Requested windows/activity/subject :",
                self.requested_samples
            )

            print(
                "Available common minimum           :",
                self.samples_per_activity_per_subject
            )

            print(
                "The lower common value will be used "
                "to preserve strict balance.\n"
            )

        # --------------------------------------------------
        # Build sampled index map
        # --------------------------------------------------
        # Each entry contains:
        #     (file_id, local_window_index)
        #
        # This map stores only selected indices and does not
        # combine all sensor windows in memory.
        # --------------------------------------------------

        self.sample_map = []

        for file_id, file_path in enumerate(
            self.files
        ):

            with np.load(
                file_path,
                allow_pickle=False
            ) as data:

                activity_labels = np.asarray(
                    data["activity"],
                    dtype=np.int64
                )

                for activity_id in range(
                    self.num_activities
                ):

                    activity_indices = (
                        np.flatnonzero(
                            activity_labels
                            == activity_id
                        )
                    )

                    selected_indices = (
                        self.rng.choice(
                            activity_indices,
                            size=(
                                self.samples_per_activity_per_subject
                            ),
                            replace=False
                        )
                    )

                    self.sample_map.extend(
                        (
                            file_id,
                            int(local_index)
                        )
                        for local_index
                        in selected_indices
                    )

        # Randomize the order of selected samples.
        self.rng.shuffle(
            self.sample_map
        )

        # --------------------------------------------------
        # Record selected subject counts
        # --------------------------------------------------

        self.selected_subject_counts = {
            class_id: 0
            for class_id in range(
                self.num_private_classes
            )
        }

        for file_path in self.files:

            class_id = file_information[
                file_path
            ]["private_value"]

            self.selected_subject_counts[
                class_id
            ] += 1

        # --------------------------------------------------
        # Dataset summary
        # --------------------------------------------------

        print("=" * 75)
        print(
            f"BALANCED {self.split.upper()} "
            f"PRIVACY DATASET"
        )
        print("=" * 75)

        print(
            "Private attribute          :",
            self.private_attribute
        )

        print(
            "Available subject counts   :",
            available_subject_counts
        )

        print(
            "Selected subject counts    :",
            self.selected_subject_counts
        )

        print(
            "Selected subject files     :",
            len(self.files)
        )

        print(
            "Activities                 :",
            self.num_activities
        )

        print(
            "Requested windows/activity :",
            self.requested_samples
        )

        print(
            "Used windows/activity/"
            "subject:",
            self.samples_per_activity_per_subject
        )

        print(
            "Total sampled windows      :",
            len(self.sample_map)
        )

        print(
            "\nDataset initialized successfully."
        )

    def __len__(self):
        """
        Return the number of selected privacy samples.
        """

        return len(
            self.sample_map
        )

    def __getitem__(
        self,
        index
    ):
        """
        Retrieve one selected IMU window and its labels.
        """

        if index < 0:
            index += len(
                self.sample_map
            )

        if (
            index < 0
            or index >= len(
                self.sample_map
            )
        ):
            raise IndexError(
                f"Dataset index out of range: {index}"
            )

        file_id, local_index = (
            self.sample_map[index]
        )

        file_path = self.files[
            file_id
        ]

        # Load only the required subject file.
        with np.load(
            file_path,
            allow_pickle=False
        ) as data:

            window = np.asarray(
                data["windows"][
                    local_index
                ],
                dtype=np.float32
            ).copy()

            activity = int(
                data["activity"][
                    local_index
                ]
            )

            gender = int(
                np.asarray(
                    data["gender"]
                ).reshape(-1)[0]
            )

            weight = int(
                np.asarray(
                    data["weight"]
                ).reshape(-1)[0]
            )

        # --------------------------------------------------
        # Validate labels
        # --------------------------------------------------

        if activity not in range(6):
            raise ValueError(
                f"Invalid activity label "
                f"{activity} in "
                f"{file_path.name}."
            )

        if gender not in range(2):
            raise ValueError(
                f"Invalid gender label "
                f"{gender} in "
                f"{file_path.name}."
            )

        if weight not in range(3):
            raise ValueError(
                f"Invalid weight label "
                f"{weight} in "
                f"{file_path.name}."
            )

        # Convert:
        # (128, 30) -> (1, 128, 30)
        window_tensor = torch.from_numpy(
            window
        ).unsqueeze(0)

        activity_tensor = torch.tensor(
            activity,
            dtype=torch.long
        )

        gender_tensor = torch.tensor(
            gender,
            dtype=torch.long
        )

        weight_tensor = torch.tensor(
            weight,
            dtype=torch.long
        )

        return (
            window_tensor,
            activity_tensor,
            gender_tensor,
            weight_tensor
        )


# ==========================================================
# Create Full Narval Privacy DataLoaders
# ==========================================================

if RUN_MODE == "full_gpu":

    # ------------------------------------------------------
    # Sampling configuration
    # ------------------------------------------------------
    # These values can be overridden using environment
    # variables in the Narval job script.
    # ------------------------------------------------------

    TRAIN_SAMPLES_PER_ACTIVITY = int(
        os.environ.get(
            "PRIVACY_TRAIN_SAMPLES_PER_ACTIVITY",
            "500"
        )
    )

    TEST_SAMPLES_PER_ACTIVITY = int(
        os.environ.get(
            "PRIVACY_TEST_SAMPLES_PER_ACTIVITY",
            "250"
        )
    )

    NUM_WORKERS = int(
        os.environ.get(
            "NUM_WORKERS",
            "2"
        )
    )

    # ------------------------------------------------------
    # Create full balanced datasets
    # ------------------------------------------------------

    full_privacy_train_dataset = (
        BalancedPrivacyIMUDataset(
            processed_dir=PROCESSED_DIR,
            split="train",
            private_attribute=(
                PRIVATE_ATTRIBUTE
            ),
            samples_per_activity_per_subject=(
                TRAIN_SAMPLES_PER_ACTIVITY
            ),
            seed=SEED
        )
    )

    full_privacy_test_dataset = (
        BalancedPrivacyIMUDataset(
            processed_dir=PROCESSED_DIR,
            split="test",
            private_attribute=(
                PRIVATE_ATTRIBUTE
            ),
            samples_per_activity_per_subject=(
                TEST_SAMPLES_PER_ACTIVITY
            ),
            seed=SEED + 1
        )
    )

    # ------------------------------------------------------
    # Create DataLoaders
    # ------------------------------------------------------

    full_privacy_train_loader = DataLoader(
        full_privacy_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
        persistent_workers=(
            NUM_WORKERS > 0
        ),
        drop_last=False
    )

    full_privacy_test_loader = DataLoader(
        full_privacy_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
        persistent_workers=(
            NUM_WORKERS > 0
        ),
        drop_last=False
    )

    # ------------------------------------------------------
    # DataLoader summary
    # ------------------------------------------------------

    print("\n" + "=" * 75)
    print("FULL NARVAL PRIVACY DATALOADERS")
    print("=" * 75)

    print(
        "Private attribute      :",
        PRIVATE_ATTRIBUTE
    )

    print(
        "Training subjects      :",
        len(
            full_privacy_train_dataset.files
        )
    )

    print(
        "Testing subjects       :",
        len(
            full_privacy_test_dataset.files
        )
    )

    print(
        "Training windows       :",
        len(
            full_privacy_train_dataset
        )
    )

    print(
        "Testing windows        :",
        len(
            full_privacy_test_dataset
        )
    )

    print(
        "Training batches       :",
        len(
            full_privacy_train_loader
        )
    )

    print(
        "Testing batches        :",
        len(
            full_privacy_test_loader
        )
    )

    print(
        "Batch size             :",
        BATCH_SIZE
    )

    print(
        "DataLoader workers     :",
        NUM_WORKERS
    )

    print(
        "Pinned memory          :",
        device.type == "cuda"
    )

    # ------------------------------------------------------
    # Verify one full training batch
    # ------------------------------------------------------

    (
        full_privacy_windows,
        full_privacy_activities,
        full_privacy_genders,
        full_privacy_weights
    ) = next(
        iter(
            full_privacy_train_loader
        )
    )

    print("\n" + "=" * 75)
    print("FULL PRIVACY BATCH VERIFICATION")
    print("=" * 75)

    print(
        "Windows shape  :",
        tuple(
            full_privacy_windows.shape
        )
    )

    print(
        "Activity shape :",
        tuple(
            full_privacy_activities.shape
        )
    )

    print(
        "Gender shape   :",
        tuple(
            full_privacy_genders.shape
        )
    )

    print(
        "Weight shape   :",
        tuple(
            full_privacy_weights.shape
        )
    )

    print(
        "Windows dtype  :",
        full_privacy_windows.dtype
    )

    print(
        "Activity dtype :",
        full_privacy_activities.dtype
    )

    print(
        "Gender dtype   :",
        full_privacy_genders.dtype
    )

    print(
        "Weight dtype   :",
        full_privacy_weights.dtype
    )

    print(
        "\nFull Narval privacy loaders "
        "verified successfully."
    )

else:

    print(
        "RUN_MODE is small_cpu. "
        "The existing balanced small privacy "
        "datasets and loaders will be used."
    )

RUN_MODE is small_cpu. The existing balanced small privacy datasets and loaders will be used.
